# WORK9 — Clean Feature Build

Workspace: `/content/drive/MyDrive/work9`. Input lineage is resolved only from the fresh Work9 Dataset V012 pointer. No Work8/Work7 run is a runtime dependency.


# WORK9 — Stage 5: Build Origin-Safe Features V0.1.3 — Calendar / Lunar / Public-Holiday Patch

**Purpose:** rebuild Pair and Branch development feature panels from the fresh **Dataset V012** with the approved Calendar V0.1.3 family.

This notebook:

- does **not** connect to Supabase;
- does **not** require a connection string;
- reads only the locked Drive artifacts;
- adds Gregorian seasonality, rule-based working-day/public-holiday proxies, Tết placement/distance, and Vietnam lunar-month features;
- builds TRAIN + OFFICIAL VALIDATION development panels only;
- deliberately excludes Frozen Test targets (`2026-04/05/06`);
- does not train models, select features, reconcile, Freeze, or publish production outputs.


In [1]:
!pip -q install 'pyarrow>=16' 'pyyaml>=6'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, yaml, hashlib, shutil, subprocess, sys, gc
from datetime import datetime, timezone
import pandas as pd

ROOT = Path('/content/drive/MyDrive/work9')
DATASET_POINTER_PATH = ROOT / '01_config' / 'current_dataset_run.json'
GATE_PATH = ROOT / '01_config' / 'work9_stage_gate_v01.yaml'
EVAL_PATH = ROOT / '01_config' / 'evaluation_contract_v02.yaml'
PAIR_CONTRACT_PATH = ROOT / '01_config' / 'pair_feature_contract_v013.yaml'
BRANCH_CONTRACT_PATH = ROOT / '01_config' / 'branch_feature_contract_v013.yaml'
CALENDAR_CONTRACT_PATH = ROOT / '01_config' / 'calendar_feature_contract_v02.yaml'
SRC_PATH = ROOT / '02_src' / 'features' / 'feature_builder_v013.py'
TEST_PATH = ROOT / '07_tests' / 'test_feature_builder_v013.py'

for p in [DATASET_POINTER_PATH, GATE_PATH, EVAL_PATH, PAIR_CONTRACT_PATH, BRANCH_CONTRACT_PATH, CALENDAR_CONTRACT_PATH, SRC_PATH, TEST_PATH]:
    assert p.exists(), f'Missing required file: {p}'

dataset_pointer = json.loads(DATASET_POINTER_PATH.read_text(encoding='utf-8'))
gate = yaml.safe_load(GATE_PATH.read_text(encoding='utf-8'))
eval_cfg = yaml.safe_load(EVAL_PATH.read_text(encoding='utf-8'))
assert dataset_pointer['status'] == 'PASS'
assert dataset_pointer['dataset_version'] == 'dataset_v012'
assert gate['authorization']['feature_build'] is True
assert gate['authorization']['frozen_test'] is False

CORE_RUN_ID = dataset_pointer['run_id']
PAIR_INPUT = Path(dataset_pointer['pair_panel_path'])
BRANCH_INPUT = Path(dataset_pointer['branch_panel_path'])
assert PAIR_INPUT.exists(), PAIR_INPUT
assert BRANCH_INPUT.exists(), BRANCH_INPUT
core_manifest_path = Path(dataset_pointer['manifest_path'])
core_manifest = json.loads(core_manifest_path.read_text(encoding='utf-8'))
assert core_manifest.get('status') == 'PASS'
assert core_manifest.get('run_id') == CORE_RUN_ID
assert core_manifest.get('dataset_version') == 'dataset_v012'
print('Current Work9 core run verified:', CORE_RUN_ID)
print('Core manifest:', core_manifest_path)
print('Frozen/Test authorization:', gate['authorization']['frozen_test'])


Mounted at /content/drive
Current Work9 core run verified: core_dataset_v012_20260815T122509Z
Core manifest: /content/drive/MyDrive/work9/06_reports/audits/core_dataset_v012_20260815T122509Z/run_manifest.json
Frozen/Test authorization: False


In [3]:
# Static/synthetic gate before touching locked data.
result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', str(TEST_PATH)],
    text=True, capture_output=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Feature builder synthetic tests failed')
print('Synthetic feature tests: PASS')


................                                                         [100%]
16 passed in 8.16s

Synthetic feature tests: PASS


In [4]:
# Import the versioned feature builder.
sys.path.insert(0, str(ROOT / '02_src' / 'features'))
import feature_builder_v013 as fb

calendar_demo = fb.build_target_calendar_table(pd.to_datetime([
    '2024-02-01', '2024-04-01', '2025-01-01', '2025-09-01',
    '2026-01-01', '2026-02-01', '2026-03-01',
]))
print('Calendar V013 sanity sample:')
print(calendar_demo[[
    'target_month','target_weekday_count','target_statutory_holiday_nominal_days',
    'target_working_days_proxy','target_lunar_month_mid','target_tet_day_of_month',
    'target_days_to_next_tet_from_midmonth','target_days_since_prev_tet_from_midmonth'
]].to_string(index=False))

pair = pd.read_parquet(PAIR_INPUT)
branch = pd.read_parquet(BRANCH_INPUT)
print('Locked Pair panel:', pair.shape)
print('Locked Branch panel:', branch.shape)
print('Pair months:', pair['month'].min(), '->', pair['month'].max())
print('Branch months:', branch['month'].min(), '->', branch['month'].max())


Calendar V013 sanity sample:
target_month  target_weekday_count  target_statutory_holiday_nominal_days  target_working_days_proxy  target_lunar_month_mid  target_tet_day_of_month  target_days_to_next_tet_from_midmonth  target_days_since_prev_tet_from_midmonth
  2024-02-01                    21                                      5                         16                       1                       10                                    349                                         5
  2024-04-01                    22                                      2                         20                       3                        0                                    289                                        65
  2025-01-01                    23                                      6                         17                      12                       29                                     14                                       340
  2025-09-01                    22             

In [5]:
# Build features strictly as-of each forecast origin.
pair_origin = fb.build_pair_origin_features(pair)
print('Pair origin features:', pair_origin.shape)

branch_origin = fb.build_branch_origin_features(branch)
print('Branch origin features:', branch_origin.shape)

train_end = eval_cfg['train']['target_end']
val_origin = eval_cfg['validation_primary']['forecast_origin']

pair_dev = fb.expand_pair_development_panel(
    pair_origin, pair,
    train_target_end=train_end,
    validation_origin=val_origin,
)
branch_dev = fb.expand_branch_development_panel(
    branch_origin, branch,
    train_target_end=train_end,
    validation_origin=val_origin,
)

# Free large intermediates before validation/write.
del pair_origin, branch_origin
gc.collect()

print('Pair development panel:', pair_dev.shape)
print('Branch development panel:', branch_dev.shape)
print('Pair split counts:')
print(pair_dev['split_role'].value_counts(dropna=False))
print('Branch split counts:')
print(branch_dev['split_role'].value_counts(dropna=False))


Pair origin features: (585614, 86)
Branch origin features: (1580, 50)
Pair development panel: (1186159, 120)
Branch development panel: (3468, 82)
Pair split counts:
split_role
TRAIN                  1107295
OFFICIAL_VALIDATION      78864
Name: count, dtype: int64
Branch split counts:
split_role
TRAIN                  3288
OFFICIAL_VALIDATION     180
Name: count, dtype: int64


In [6]:
validation = fb.validate_feature_stage(pair_dev, branch_dev)
print(json.dumps(validation, ensure_ascii=False, indent=2, default=str))
if validation['status'] != 'PASS':
    raise RuntimeError('FEATURE BUILD VALIDATION FAILED — outputs will not be published')

inventory = fb.build_feature_inventory(pair_dev, branch_dev)
print('Feature validation: PASS')
print('Inventory rows:', len(inventory))
print('Initial Pair candidates:', int(inventory.query("track == 'PAIR' and initial_model_candidate == True").shape[0]))
print('Initial Branch candidates:', int(inventory.query("track == 'BRANCH' and initial_model_candidate == True").shape[0]))


{
  "status": "PASS",
  "checks": {
    "pair_grain_unique": {
      "pass": true,
      "detail": null
    },
    "branch_grain_unique": {
      "pass": true,
      "detail": null
    },
    "pair_information_origin_safe": {
      "pass": true,
      "detail": null
    },
    "branch_information_origin_safe": {
      "pass": true,
      "detail": null
    },
    "pair_target_horizon_alignment": {
      "pass": true,
      "detail": null
    },
    "branch_target_horizon_alignment": {
      "pass": true,
      "detail": null
    },
    "frozen_test_not_touched_pair": {
      "pass": true,
      "detail": null
    },
    "frozen_test_not_touched_branch": {
      "pass": true,
      "detail": null
    },
    "pair_train_cutoff_respected": {
      "pass": true,
      "detail": null
    },
    "branch_train_cutoff_respected": {
      "pass": true,
      "detail": null
    },
    "pair_validation_origin_locked": {
      "pass": true,
      "detail": null
    },
    "branch_validation_origin

In [7]:
# Publish immutable Stage-5 artifacts only after validation PASS.
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

RUN_ID = 'feature_stage_v013_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DATA_OUT = ROOT / '04_data' / 'features' / RUN_ID
REPORT_OUT = ROOT / '06_reports' / 'feature_validation' / RUN_ID
RUN_OUT = ROOT / '08_runs' / RUN_ID
for p in [DATA_OUT, REPORT_OUT, RUN_OUT]:
    p.mkdir(parents=True, exist_ok=False)

pair_path = DATA_OUT / 'pair_feature_panel_dev_v013.parquet'
branch_path = DATA_OUT / 'branch_feature_panel_dev_v013.parquet'
inv_path = REPORT_OUT / 'feature_inventory.csv'
val_path = REPORT_OUT / 'feature_validation.json'
profile_path = REPORT_OUT / 'feature_profile.json'
manifest_path = RUN_OUT / 'run_manifest.json'

pair_dev['run_id'] = RUN_ID
branch_dev['run_id'] = RUN_ID

pair_dev.to_parquet(pair_path, index=False)
branch_dev.to_parquet(branch_path, index=False)
inventory.to_csv(inv_path, index=False, encoding='utf-8-sig')
val_path.write_text(json.dumps(validation, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

output_sha256 = {
    'pair_feature_panel': sha256_file(pair_path),
    'branch_feature_panel': sha256_file(branch_path),
    'feature_inventory': sha256_file(inv_path),
}

profile = {
    'run_id': RUN_ID,
    'pair_rows': int(len(pair_dev)),
    'branch_rows': int(len(branch_dev)),
    'pair_train_rows_total': int(pair_dev['split_role'].eq('TRAIN').sum()),
    'pair_train_rows_eligible': int(pair_dev['historical_train_mask'].sum()),
    'pair_validation_rows_total': int(pair_dev['split_role'].eq('OFFICIAL_VALIDATION').sum()),
    'pair_validation_rows_eligible': int(pair_dev['official_validation_mask'].sum()),
    'branch_train_rows_total': int(branch_dev['split_role'].eq('TRAIN').sum()),
    'branch_train_rows_eligible': int(branch_dev['historical_train_mask'].sum()),
    'branch_validation_rows_total': int(branch_dev['split_role'].eq('OFFICIAL_VALIDATION').sum()),
    'branch_validation_rows_eligible': int(branch_dev['official_validation_mask'].sum()),
    'pair_target_min': str(pair_dev['target_month'].min()),
    'pair_target_max': str(pair_dev['target_month'].max()),
    'branch_target_min': str(branch_dev['target_month'].min()),
    'branch_target_max': str(branch_dev['target_month'].max()),
    'frozen_test_touched': False,
}
profile_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding='utf-8')

manifest = {
    'run_id': RUN_ID,
    'run_type': 'FEATURE_STAGE_V013',
    'status': 'PASS',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'locked_core_run_id': CORE_RUN_ID,
    'dataset_version': core_manifest['dataset_version'],
    'pair_feature_version': fb.PAIR_FEATURE_VERSION,
    'branch_feature_version': fb.BRANCH_FEATURE_VERSION,
    'code_sha256': sha256_file(SRC_PATH),
    'evaluation_contract_sha256': sha256_file(EVAL_PATH),
    'pair_feature_contract_sha256': sha256_file(PAIR_CONTRACT_PATH),
    'branch_feature_contract_sha256': sha256_file(BRANCH_CONTRACT_PATH),
    'calendar_feature_contract_sha256': sha256_file(CALENDAR_CONTRACT_PATH),
    'inputs': {
        'pair_panel': str(PAIR_INPUT),
        'branch_panel': str(BRANCH_INPUT),
    },
    'outputs': {
        'pair_feature_panel': str(pair_path),
        'branch_feature_panel': str(branch_path),
        'feature_inventory': str(inv_path),
        'feature_validation': str(val_path),
        'feature_profile': str(profile_path),
    },
    'output_sha256': output_sha256,
    'row_counts': {
        'pair': int(len(pair_dev)),
        'branch': int(len(branch_dev)),
    },
    'safety': {
        'supabase_accessed': False,
        'frozen_test_touched': False,
        'feature_selection_run': False,
        'model_training_run': False,
        'production_published': False,
    },
    'calendar_v013': {
        'working_days_semantic': 'RULE_BASED_PROXY_NOT_COMPANY_SHUTDOWN',
        'public_sector_swap_schedule_used_as_model_feature': False,
        'company_shutdown_calendar_used': False,
    },
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

# Preserve the contracts actually used for this run.
contract_out = RUN_OUT / 'contracts'
contract_out.mkdir()
for src in [DATASET_POINTER_PATH, GATE_PATH, EVAL_PATH, PAIR_CONTRACT_PATH, BRANCH_CONTRACT_PATH, CALENDAR_CONTRACT_PATH]:
    shutil.copy2(src, contract_out / src.name)

print('FEATURE STAGE PASS')
print('RUN_ID:', RUN_ID)
print('Pair feature panel:', pair_path)
print('Branch feature panel:', branch_path)
print('Validation:', val_path)
print('Frozen Test touched: FALSE')
print('Output SHA256 recorded in manifest and current feature pointer.')

# Work9 pointer consumed automatically by Feature Selection.
current_feature = {
    'run_id': RUN_ID,
    'status': 'PASS',
    'dataset_run_id': CORE_RUN_ID,
    'pair_feature_version': fb.PAIR_FEATURE_VERSION,
    'branch_feature_version': fb.BRANCH_FEATURE_VERSION,
    'pair_feature_panel_path': str(pair_path),
    'branch_feature_panel_path': str(branch_path),
    'feature_inventory_path': str(inv_path),
    'feature_manifest_path': str(manifest_path),
    'output_sha256': output_sha256,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}
(ROOT/'01_config'/'current_feature_run.json').write_text(json.dumps(current_feature, indent=2), encoding='utf-8')
print('Current feature pointer updated:', ROOT/'01_config'/'current_feature_run.json')


FEATURE STAGE PASS
RUN_ID: feature_stage_v013_20260815T123431Z
Pair feature panel: /content/drive/MyDrive/work9/04_data/features/feature_stage_v013_20260815T123431Z/pair_feature_panel_dev_v013.parquet
Branch feature panel: /content/drive/MyDrive/work9/04_data/features/feature_stage_v013_20260815T123431Z/branch_feature_panel_dev_v013.parquet
Validation: /content/drive/MyDrive/work9/06_reports/feature_validation/feature_stage_v013_20260815T123431Z/feature_validation.json
Frozen Test touched: FALSE
Output SHA256 recorded in manifest and current feature pointer.
Current feature pointer updated: /content/drive/MyDrive/work9/01_config/current_feature_run.json


## Stop here

A PASS from this notebook authorizes **audit of Feature V0.1.3 only**.

After PASS, continue only to the fresh Work9 Feature Selection V0.4 stage. No old feature-selection/model/freeze artifact may be reused. Frozen Test remains blocked.
